# Simulation Service

If all services are not already started, open a shell and run:
```sh
python -m startup.start_all_services
```

#### 1. Listen for state messages

In [ ]:
from threading import Thread
from communication import protocol
from communication.rabbitmq import Rabbitmq

def receiver() -> None:
    def on_state_message_received(channel, method, properties, body) -> None:
        print(body)

    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    rmq.subscribe(protocol.ROUTING_KEY_SIM_STATE, on_state_message_received)
    rmq.start_consuming()


Thread(target=receiver).start()

#### 2. Sending simulation control message

In [ ]:
import numpy as np
from communication import protocol
from communication.rabbitmq import Rabbitmq

try:
    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    print("✓ Connected to RabbitMQ successfully")
except Exception as e:
    print(f"✗ Failed to connect to RabbitMQ: {e}")
    print("\nMake sure RabbitMQ is running. You can start it with:")
    print("  python -m startup.start_docker_rabbitmq")

In [ ]:
rmq.send_message(
    routing_key=protocol.ROUTING_KEY_SIM_CTRL,
    message={
        protocol.SimMsgKeys.TYPE: protocol.SimMsgFields.POSITION,
        protocol.SimMsgKeys.ACTUAL_JOINT_POSITIONS: [0, 0, 0, 0, 0, 0],
        protocol.SimMsgKeys.TARGET_JOINT_POSITIONS: [0, np.pi / 2, 0, 0, 0, 0],
    },
)